# Classical Reasoning Agent (Natural Language KB → Datalog-style KB → Forward/Backward Chaining → True/False)
**Pipeline:** rule-based NL parser → facts + Horn rules (with negation) → stratified forward chaining (main engine, with proof traces)
and SLD backward chaining with unification (second engine). No generative AI is used for any decision.

In [1]:
import re, time, os, sys, json, itertools, collections
from collections import defaultdict, namedtuple
sys.setrecursionlimit(20000)

# ---------------- CONFIG (edit these) ----------------
FOL_STRATEGY = "astar"             # clause selection for resolution: bfs | dfs | ucs | astar
FOL_TIME_LIMIT = 2.0               # seconds per first-order proof attempt
SEMANTICS = "cwa"                  # "cwa": closed-world, negation = failure (matches RuleTaker samples); "owa": explicit/strong negation only
ENGINE = "fc"                       # "fc" (forward chaining) or "bc" (backward chaining)
PRIVATE_TEST_PATH = "private_test.csv"   # hidden test file (csv/json/jsonl/parquet). Columns are detected automatically (see section 5)
SUBMISSION_PATH = "submission.csv"
MAX_EVAL_PER_DATASET = None         # None = evaluate every public example (they are small); set an int to subsample
# HuggingFace datasets: (short name, repo id, config). Wrong/unavailable repos are skipped automatically.
DATA_DIR = "datasets"              # optional local copy of the public data; any file missing here is downloaded from Hugging Face automatically
DEV_FILES = {
    "RuleTaker-train": "ruletaker/train.parquet", "RuleTaker-test": "ruletaker/test.parquet", "RuleTaker-ood": "ruletaker/ood_test.parquet",
    "FOLIO-train": "FOLIO/folio_v2_train.jsonl", "FOLIO-validation": "FOLIO/folio_v2_validation.jsonl",
    "CounterBench-alpha": "counterbench/data_balanced_alpha_V1.json", "CounterBench-backdoor": "counterbench/data_balanced_backdoor_V2.json",
}
SEED = 0
# Local files can also be evaluated: {"name": "path"}

## 1. Internal representation
* **Literal** `Lit(neg, pred, args)`; variables start with `?`. Unary predicates = properties (`young(bob)`), binary = relations (`chase(cat, dog)`).
* **Rule** `Rule(body, head, text)` = Horn clause. Facts are ground atoms.
* **Negation:** if the KB contains *explicit* negative facts / negative rule heads → open-world: `not p` becomes the separate atom `~p` (strong negation).
  Otherwise → closed-world: `not` in rule bodies / queries is negation-as-failure, evaluated with **stratified** forward chaining.

In [2]:
Lit = namedtuple("Lit", "neg pred args")
Rule = namedtuple("Rule", "body head text")
def is_var(t): return isinstance(t, str) and t.startswith("?")

DET = {"the", "a", "an"}
PRON = {"they", "it", "he", "she", "them", "him", "her"}
QVAR = {"someone", "something", "somebody", "anyone", "anything", "anybody", "everyone", "everything", "everybody"}
GEN_NOUN = {"people", "things", "person", "thing", "ones", "persons", "individuals"}
COP = {"is", "are", "was", "were", "isn't", "aren't"}
AUX = {"does", "do", "did", "doesn't", "don't"}
IRREG = {"has": "have", "does": "do", "goes": "go", "is": "be"}

def tokenize(s):
    return re.findall(r"[A-Za-z0-9_'’\-]+", s.replace("’", "'"))

def norm_verb(w):
    if w in IRREG: return IRREG[w]
    if w.endswith("ies") and len(w) > 4: return w[:-3] + "y"
    if re.search(r"(ches|shes|sses|xes|zes)$", w): return w[:-2]
    if w.endswith("s") and not w.endswith("ss"): return w[:-1]
    return w

def singular(w):
    if w.endswith("ies") and len(w) > 4: return w[:-3] + "y"
    if w.endswith("s") and not w.endswith("ss") and len(w) > 3: return w[:-1]
    return w

def ent(tokens, role="subj"):
    toks = [t for t in tokens if t.lower() not in DET]
    if not toks: return None
    if len(toks) == 1:
        w = toks[0].lower()
        if w in PRON: return "?x"
        if w in QVAR: return "?x" if role == "subj" else "?y"
    return "_".join(t.lower() for t in toks)

def pred_name(toks):
    toks = [t.lower() for t in toks]
    while toks and toks[0] in DET: toks = toks[1:]
    return "_".join(toks) if toks else None

def find_split(toks, allow_base=False):
    low = [t.lower() for t in toks]; n = len(toks)
    for i in range(1, n):
        w = low[i]
        if w in COP: return i, "cop"
        if w in AUX: return i, "aux"
        if (i + 1 < n and w.endswith("s") and w != "is" and
                (low[i + 1] in DET or toks[i + 1][0].isupper() or low[i + 1] in PRON | QVAR)):
            return i, "verb"
    if low[0] in PRON | QVAR and n > 1: return 1, "verb_base"
    if allow_base and n > 1: return 1, "verb_base"
    return None

def parse_full(toks, allow_base=False):
    sp = find_split(toks, allow_base)
    if not sp: return None
    i, kind = sp
    subj = ent(toks[:i], "subj")
    if subj is None: return None
    low = [t.lower() for t in toks]
    if kind == "cop":
        neg = low[i] in ("isn't", "aren't")
        rest = toks[i + 1:]
        if rest and rest[0].lower() == "not": neg, rest = True, rest[1:]
        p = pred_name(rest)
        if not p: return None
        return [Lit(neg, p, (subj,))], ("cop", subj)
    neg = False
    if kind == "aux":
        neg = low[i] in ("doesn't", "don't")
        j = i + 1
        if j < len(toks) and low[j] == "not": neg, j = True, j + 1
        if j >= len(toks): return None
        verb, rest = low[j], toks[j + 1:]
    elif kind == "verb":
        verb, rest = norm_verb(low[i]), toks[i + 1:]
    else:
        verb, rest = low[i], toks[i + 1:]
        if rest and rest[0].lower() == "not": neg, rest = True, rest[1:]
    obj = ent(rest, "obj") if rest else None
    args = (subj,) if obj is None else (subj, obj)
    return [Lit(neg, verb, args)], ("verb", subj, verb, neg)

def parse_cont(toks, state):
    if state[0] == "cop":
        neg = False
        if toks and toks[0].lower() == "not": neg, toks = True, toks[1:]
        p = pred_name(toks)
        return [Lit(neg, p, (state[1],))] if p else None
    obj = ent(toks, "obj")
    return [Lit(state[3], state[2], (state[1], obj))] if obj else None

def prop_fallback(toks):
    low = [t.lower() for t in toks if t.lower() not in DET]
    neg = False
    if low and low[0] == "not": neg, low = True, low[1:]
    return [Lit(neg, "prop_" + "_".join(low), ())]

def parse_conj(text, allow_base=False):
    pieces = [p for p in re.split(r"\s*,\s*and\s+|\s+and\s+|\s*,\s*", text.strip()) if p.strip()]
    lits, state = [], None
    for p in pieces:
        toks = tokenize(p)
        if not toks: continue
        full = parse_full(toks, allow_base and state is None)
        if full: l, state = full; lits += l; continue
        if state:
            c = parse_cont(toks, state)
            if c: lits += c; continue
        lits += prop_fallback(toks)
    return lits

def has_var(lits): return any(is_var(a) for l in lits for a in l.args)

def parse_sentence(s):
    """Returns ('fact', [Lit]) or ('rule', [Rule])."""
    orig = s.strip(); s = orig.rstrip(".!?").strip()
    if not s: return None
    m = re.match(r"^(?:if|when|whenever)\s+(.*?)\s*,?\s+then\s+(.*)$", s, re.I) or \
        re.match(r"^(?:if|when|whenever)\s+([^,]*),\s*(.*)$", s, re.I)
    if m:
        body, head = parse_conj(m.group(1)), parse_conj(m.group(2))
        return "rule", [Rule(tuple(body), h, orig) for h in head]
    toks = tokenize(s); low = [t.lower() for t in toks]
    # universal statements: "All/Every X are Y", "Big, red things are nice", "Young people are kind"
    subj_end = None; body_start = 0; plural_noun = False
    if low and low[0] in ("all", "every", "each", "any") and len(toks) > 2:
        sp = find_split(toks[1:], allow_base=True)
        if sp: subj_end, body_start, plural_noun = sp[0] + 1, 1, True
    else:
        for g, w in enumerate(low):
            if w in COP or w in AUX: break
            if w in GEN_NOUN and g < len(toks) - 1: subj_end = g + 1; break
    if subj_end:
        subj_toks = toks[body_start:subj_end]
        if subj_toks and subj_toks[-1].lower() in GEN_NOUN: subj_toks = subj_toks[:-1]
        elif plural_noun and subj_toks: subj_toks = subj_toks[:-1] + [singular(subj_toks[-1].lower())]
        body, neg = [], False
        for t in subj_toks:                     # RuleTaker adjectives are single words ("Big, red things ...")
            w = t.lower()
            if w == "and" or w in DET: continue
            if w == "not": neg = True; continue
            body.append(Lit(neg, w, ("?x",))); neg = False
        head = parse_conj("they " + " ".join(toks[subj_end:]))
        return "rule", [Rule(tuple(body), h, orig) for h in head]
    lits = parse_conj(s)
    if has_var(lits): return "rule", [Rule((), h, orig) for h in lits]   # e.g. "Everything is big"
    return "fact", lits

def split_sentences(context):
    if isinstance(context, (list, tuple)): context = "\n".join(map(str, context))
    out = []
    for line in str(context).split("\n"):
        for s in re.split(r"(?<=[.!?])\s+", line.strip()):
            if s.strip(): out.append(s.strip())
    return out


# ---- "Facts:/Rules:" template format (course RuleTaker variant: ground facts + paraphrased implications) ----
LEAD = r"(?:when|given that|based on|assuming|since|because|if|provided that|suppose)"
CONC = r"(?:then|we can conclude(?: that)?|it follows that|it is evident that|we can say(?: that)?|we can see that|we know that|this means(?: that)?|therefore)"
TEMPLATE_RULE = re.compile(rf"^{LEAD}\s+(.*?)\s*,\s*{CONC}\s+(.*?)\s*[.!?]?\s*$", re.I)
def is_template_context(ctx): return bool(re.search(r"^\s*facts?\s*:", str(ctx), re.I | re.M))

def parse_template_context(ctx):
    """Everything is lower-cased first, so a clause means the same thing in a fact, a rule and the query."""
    facts, rules, failed, sect = [], [], [], None
    for line in str(ctx).split("\n"):
        line = line.strip()
        if not line: continue
        if re.match(r"^facts?\s*:", line, re.I): sect = "f"; continue
        if re.match(r"^rules?\s*:", line, re.I): sect = "r"; continue
        low = line.lower()
        if sect == "r":
            m = TEMPLATE_RULE.match(low)
            if not m: failed.append(line); continue
            body, head = parse_conj(m.group(1)), parse_conj(m.group(2))
            rules.extend(Rule(tuple(body), h, line) for h in head)
        else:
            facts.extend(parse_conj(low.rstrip(".!? ")))
    return facts, rules, failed

def parse_context(context):
    if is_template_context(context):
        f, r, _ = parse_template_context(context); return f, r
    facts, rules = [], []
    for s in split_sentences(context):
        r = parse_sentence(s)
        if not r: continue
        (facts.extend if r[0] == "fact" else rules.extend)(r[1])
    return facts, rules

def parse_query(q):
    q = re.sub(r"^\s*(query|question|statement|hypothesis|conclusion)\s*:\s*", "", str(q), flags=re.I)
    q = re.sub(r"^\s*(true or false|is it true that|is the following true)\s*[:?]?\s*", "", q, flags=re.I)
    return parse_conj(q.strip().rstrip(".!?"))

## 2. Reasoning engines
**Forward chaining** (main): relevance pruning by predicate dependency graph → stratification (for NAF) → per-stratum fixpoint with indexed joins and unification (matching) → provenance for traces; early stop once a positive goal is derived.
**Backward chaining**: SLD resolution with unification, variable standardising-apart, ancestor loop check, depth bound and NAF.

In [3]:
def fmt(pred, args): return f"{pred}({', '.join(a.replace('_',' ') for a in args)})" if args else pred

def prune(facts, rules, goal_preds):
    deps = defaultdict(set)
    for r in rules: deps[r.head.pred] |= {l.pred for l in r.body}
    rel, stack = set(goal_preds), list(goal_preds)
    while stack:
        p = stack.pop()
        for q in deps.get(p, ()):
            if q not in rel: rel.add(q); stack.append(q)
    return [f for f in facts if f[0] in rel], [r for r in rules if r.head.pred in rel]

def stratify(rules):
    preds = set()
    for r in rules: preds.add(r.head.pred); preds |= {l.pred for l in r.body}
    s = {p: 0 for p in preds}
    for _ in range(len(preds) + 1):
        ch = False
        for r in rules:
            for l in r.body:
                need = s[l.pred] + (1 if l.neg else 0)
                if s[r.head.pred] < need: s[r.head.pred] = need; ch = True
        if not ch: break
    return s

class ForwardChainer:
    def __init__(self, facts, rules, domain=()):
        self.db = defaultdict(set); self.idx = defaultdict(set); self.prov = {}
        self.rules = list(rules)
        dom = set(domain)
        for p, a in facts:
            self._add(p, a, None); dom |= set(a)
        for r in rules:
            for l in list(r.body) + [r.head]: dom |= {a for a in l.args if not is_var(a)}
        self.domain = sorted(dom)
        self.compiled = [self._compile(i, r) for i, r in enumerate(rules)]

    def _add(self, pred, args, prov):
        if args in self.db[pred]: return False
        self.db[pred].add(args)
        for j, a in enumerate(args): self.idx[(pred, j, a)].add(args)
        if prov is not None: self.prov[(pred, args)] = prov
        return True

    def _compile(self, i, r):
        pos = [l for l in r.body if not l.neg]; neg = [l for l in r.body if l.neg]
        order, bound = [], set()
        while pos:
            def score(l):
                b = sum(1 for a in l.args if (not is_var(a)) or a in bound)
                return (-b, len(self.db.get(l.pred, ())))
            nxt = min(pos, key=score); pos.remove(nxt); order.append(nxt)
            bound |= {a for a in nxt.args if is_var(a)}
        return (i, order, neg, r.head)

    def _cands(self, pred, args):
        best = None
        for j, a in enumerate(args):
            if not is_var(a):
                s = self.idx.get((pred, j, a))
                if s is None: return ()
                if best is None or len(s) < len(best): best = s
        return best if best is not None else self.db.get(pred, ())

    def _solve(self, lits, i, th):
        if i == len(lits): yield th; return
        l = lits[i]
        args = tuple(th.get(a, a) if is_var(a) else a for a in l.args)
        for tup in self._cands(l.pred, args):
            new, ok = th, True
            for a, v in zip(args, tup):
                if is_var(a):
                    if new is th: new = dict(th)
                    if a in new:
                        if new[a] != v: ok = False; break
                    else: new[a] = v
                elif a != v: ok = False; break
            if ok: yield from self._solve(lits, i + 1, new)

    def _negs_ok(self, negs, th):
        for l in negs:
            args = tuple(th.get(a, a) if is_var(a) else a for a in l.args)
            if any(is_var(a) for a in args):
                if any(True for _ in self._solve([Lit(False, l.pred, args)], 0, {})): return False
            elif args in self.db.get(l.pred, ()): return False
        return True

    def _fire(self, cr):
        i, pos, neg, head = cr
        derived = []
        nvars = sorted({a for l in neg for a in l.args if is_var(a)})
        for th0 in self._solve(pos, 0, {}):
            free_n = [v for v in nvars if v not in th0]          # variables bound only by negative literals -> range over domain
            for combo in (itertools.product(self.domain, repeat=len(free_n)) if free_n else [()]):
                th = {**th0, **dict(zip(free_n, combo))} if free_n else th0
                if neg and not self._negs_ok(neg, th): continue
                self._emit(th, pos, neg, head, i, derived)
        c = 0
        for p, a, pv in derived: c += self._add(p, a, pv)
        return c

    def _emit(self, th, pos, neg, head, i, derived):
        if True:
            hargs = tuple(th.get(a, a) if is_var(a) else a for a in head.args)
            free = sorted({a for a in hargs if is_var(a)})
            combos = [dict(zip(free, c)) for c in itertools.product(self.domain, repeat=len(free))] if free else [{}]
            for m in combos:
                ga = tuple(m.get(a, a) for a in hargs)
                if ga in self.db.get(head.pred, ()): continue
                sup = [(l.pred, tuple(th.get(a, a) if is_var(a) else a for a in l.args)) for l in pos]
                nsup = [(l.pred, tuple(th.get(a, a) if is_var(a) else a for a in l.args)) for l in neg]
                derived.append((head.pred, ga, (i, sup, nsup)))

    def run(self, goal=None):
        st = stratify(self.rules)
        groups = defaultdict(list)
        for cr in self.compiled: groups[st.get(cr[3].pred, 0)].append(cr)
        for k in sorted(groups):
            while True:
                n = sum(self._fire(cr) for cr in groups[k])
                if goal and goal[1] in self.db.get(goal[0], ()): return
                if not n: break

    def holds(self, lit):
        present = lit.args in self.db.get(lit.pred, ())
        return (not present) if lit.neg else present

    def explain(self, fact, indent=0, seen=None):
        seen = set() if seen is None else seen
        pad = "  " * indent; pred, args = fact
        pv = self.prov.get(fact)
        if pv is None: return [f"{pad}{fmt(pred,args)}   [given fact]"]
        if fact in seen: return [f"{pad}{fmt(pred,args)}   [see above]"]
        seen.add(fact)
        i, sup, nsup = pv
        out = [f"{pad}{fmt(pred,args)}   [by rule: {self.rules[i].text}]"]
        for s in sup: out += self.explain(s, indent + 1, seen)
        for p, a in nsup: out.append(f"{pad}  not {fmt(p,a)}   [negation as failure: not derivable]")
        return out

class BackwardChainer:
    def __init__(self, facts, rules, max_depth=30):
        self.facts = defaultdict(set); self.rules = defaultdict(list)
        for p, a in facts: self.facts[p].add(a)
        for r in rules: self.rules[r.head.pred].append(r)
        self.max_depth = max_depth; self.cnt = 0
    def walk(self, t, th):
        while is_var(t) and t in th: t = th[t]
        return t
    def unify(self, a, b, th):
        if len(a) != len(b): return None
        th = dict(th)
        for x, y in zip(a, b):
            x, y = self.walk(x, th), self.walk(y, th)
            if x == y: continue
            if is_var(x): th[x] = y
            elif is_var(y): th[y] = x
            else: return None
        return th
    def rename(self, r):
        self.cnt += 1; s = f"#{self.cnt}"
        f = lambda a: tuple(x + s if is_var(x) else x for x in a)
        body = [Lit(l.neg, l.pred, f(l.args)) for l in r.body]
        body = [l for l in body if not l.neg] + [l for l in body if l.neg]   # NAF goals last
        return body, Lit(r.head.neg, r.head.pred, f(r.head.args))
    def solve(self, goals, th, depth):
        if not goals: yield th; return
        (g, stack), rest = goals[0], goals[1:]
        args = tuple(self.walk(a, th) for a in g.args)
        if g.neg:
            if not any(True for _ in self.solve(((Lit(False, g.pred, args), frozenset()),), th, depth + 1)):
                yield from self.solve(rest, th, depth)
            return
        key = (g.pred, args)
        if key in stack or depth > self.max_depth: return
        st2 = stack | {key}
        for tup in self.facts.get(g.pred, ()):
            t2 = self.unify(args, tup, th)
            if t2 is not None: yield from self.solve(rest, t2, depth)
        for r in self.rules.get(g.pred, ()):
            body, head = self.rename(r)
            t2 = self.unify(args, head.args, th)
            if t2 is not None: yield from self.solve(tuple((b, st2) for b in body) + rest, t2, depth + 1)
    def prove(self, lits):
        return any(True for _ in self.solve(tuple((l, frozenset()) for l in lits), {}, 0))

### 2b. CounterBench module (deterministic causal models → Horn rules → the same forward chainer)
"A causes B", "A and B together cause C", "A or B causes C", "not A causes B", "A causes not B", "A causes B, C and D".
Each variable becomes `occurs(v)`; negation = NAF (the graph is a DAG, so the program is stratified).
`do(v=0)` removes every rule that derives `v`; `do(v=1)` removes them and adds the fact. Observed roots become facts.

In [4]:
def _cb_lit(txt):
    t = txt.strip().strip(".?,").strip()
    neg = False
    if t.lower().startswith("not "): neg, t = True, t[4:].strip()
    return neg, t

def looks_like_counterbench(text):
    return bool(re.search(r"\bcauses?\b", text)) and bool(re.search(r"\bwould\b.*\boccur\b", text, re.I | re.S))

def parse_counterbench(text):
    """-> dict(rules, target, ints, evidence, priors, domain) or None."""
    m = re.search(r"We know that\s+(.*?)\.(?=\s|$)", text, re.S)
    if not m: return None
    segs, stm = [x.strip() for x in re.split(r",\s*", m.group(1)) if x.strip()], []
    for sg in segs:
        sg = re.sub(r"^and\s+", "", sg)
        if re.search(r"\bcauses?\b", sg) or not stm: stm.append(sg)
        else: stm[-1] += ", " + sg                       # continuation of a target list: "X causes A, B and C"
    rules, domain = [], set()
    for st in stm:
        mm = re.match(r"(.*?)\s+causes?\s+(.*)$", st)
        if not mm: return None
        lhs = re.sub(r"\s+together$", "", mm.group(1).strip())
        dnf = [[_cb_lit(l) for l in re.split(r"\s+and\s+", clause)] for clause in re.split(r"\s+or\s+", lhs)]
        for tg in re.split(r"\s*,\s*|\s+and\s+", mm.group(2)):
            if not tg.strip(): continue
            tneg, tname = _cb_lit(tg); domain.add(tname)
            d = dnf
            if tneg:                                       # De Morgan: NOT(OR_i AND_j l_ij) = AND_i OR_j NOT l_ij  -> DNF by product
                d = [list(c) for c in itertools.product(*[[(not n, v) for n, v in cl] for cl in dnf])]
            for cl in d:
                for n, v in cl: domain.add(v)
                rules.append(Rule(tuple(Lit(n, "occurs", (v,)) for n, v in cl), Lit(False, "occurs", (tname,)), st))
    mt = re.search(r"[Ww]ould\s+(.*?)\s+occur", text)
    if not mt or not rules: return None
    target = _cb_lit(mt.group(1))[1]; domain.add(target)
    ints, evidence = [], []
    mi = re.search(r"occur\s+if\s+(.*?)(\s+instead of\s+(.*?))?\?", text, re.S)
    if mi:
        for l in re.split(r"\s+and\s+", mi.group(1)):
            n, v = _cb_lit(l); ints.append((v, 0 if n else 1))
        if mi.group(3):                                    # "... instead of X": X actually occurred in the factual world
            for l in re.split(r"\s+and\s+", mi.group(3)):
                n, v = _cb_lit(l); evidence.append((v, 0 if n else 1))
    for kw in ("[Aa]ssume", "suppose"):
        for mm in re.finditer(kw + r"\s+((?:not\s+)?[^,.?]+)", text):
            n, v = _cb_lit(mm.group(1)); ints.append((v, 0 if n else 1))
    for mm in re.finditer(r"[Ww]e observed\s+([^.?]+?)(?=\s+[Ww]ould\b|[.?]|$)", text):
        for l in re.split(r"\s+and\s+|,\s*", mm.group(1)):
            n, v = _cb_lit(l); evidence.append((v, 0 if n else 1)); domain.add(v)
    priors = {v: float(p) for v, p in re.findall(r"(\w+)\s*~\s*Bern\(\s*([0-9.]+)\s*\)", text)}
    return dict(rules=rules, target=target, ints=ints, evidence=evidence, priors=priors, domain=domain)

def _cb_world(rules, true_vars, domain):
    fc = ForwardChainer({("occurs", (v,)) for v in true_vars}, rules, domain); fc.run()
    return fc, {a[0] for a in fc.db.get("occurs", ())}

def answer_counterbench(text, trace=False):
    """Counterfactual query on a deterministic SCM: abduction (enumerate exogenous worlds consistent with the evidence,
    weighted by Bernoulli priors) -> action (graph surgery) -> prediction (forward chaining). Yes iff P(target | evidence) > 1/2."""
    p = parse_counterbench(text)
    if p is None: return None
    rules, target, ints, ev, priors, domain = p["rules"], p["target"], p["ints"], p["evidence"], p["priors"], p["domain"]
    heads = {r.head.args[0] for r in rules}; roots = sorted(domain - heads)
    cf_rules = list(rules)
    for v, _ in ints: cf_rules = [r for r in cf_rules if r.head.args[0] != v]
    num = den = 0.0; best = None
    for bits in itertools.product((0, 1), repeat=len(roots)):
        w = 1.0
        for r_, b in zip(roots, bits): pr = priors.get(r_, 0.5); w *= pr if b else 1 - pr
        if w == 0: continue
        on = {r_ for r_, b in zip(roots, bits) if b}
        _, fact = _cb_world(rules, on, domain)                                   # abduction: factual world must match the evidence
        if any((v in fact) != bool(val) for v, val in ev): continue
        cf_on = (on - {v for v, _ in ints}) | {v for v, val in ints if val == 1}  # action
        fc2, cf = _cb_world(cf_rules, cf_on, domain)                              # prediction
        den += w; hit = target in cf; num += w * hit
        if best is None or (hit and not best[0]): best = (hit, fc2, on)
    if den == 0:                                                                  # inconsistent evidence: ignore the evidence on non-roots
        fc2, cf = _cb_world(cf_rules, {v for v, val in ints if val == 1}, domain); res = target in cf; best = (res, fc2, set())
    else: res = num / den > 0.5
    if not trace: return res
    lines = [f"exogenous variables: {roots}, priors: {priors or 'uniform 0.5'}", f"evidence: {ev}", f"action: {['do(%s=%d)' % x for x in ints]}"]
    if den: lines.append(f"posterior P({target} | evidence, do) = {num/den:.3f}")
    lines += best[1].explain(("occurs", (target,))) if (best and best[0] and target in {a[0] for a in best[1].db.get('occurs', ())}) else [f"occurs({target}) is not derivable in the counterfactual world -> No"]
    return res, lines

### 2c. First-order logic module (FOLIO: formulas are given directly)
Pipeline: tolerant FOL parser (Unicode + ASCII) → negation normal form → standardise apart + **Skolemisation** → **CNF** →
**resolution** (binary resolution + factoring, **unification with occurs check**, set-of-support, tautology/duplicate/unit-subsumption pruning).
The given-clause selection strategy is a parameter: `bfs` (age), `dfs` (LIFO), `ucs`/`best` (clause weight), `astar` (depth + literal count).
`premises ∧ ¬conclusion` unsatisfiable ⇒ **entailed ⇒ True**; otherwise (saturated or resource limit) ⇒ False.

In [5]:
import heapq
_ASCII = [(r"<->", "↔"), (r"—>|–>|⟶", "→"), (r"<=>", "↔"), (r"->", "→"), (r"=>", "→"), (r"\bforall\b|\\forall|\ball\b", "∀"), (r"\bexists\b|\\exists", "∃"),
          (r"&|\band\b|\\land|\\wedge", "∧"), (r"\||\bor\b|\\lor|\\vee", "∨"), (r"~|!|\bnot\b|\\neg|\\lnot", "¬"), (r"\bxor\b", "⊕"), ("⟷", "↔"), ("≠", "≠")]
_FOL_TOK = re.compile(r"\s*(?:(∀|∃|∧|∨|¬|→|↔|⊕|≠|=|\(|\)|,|\.|:)|((?:[^\s∀∃∧∨¬→↔⊕≠=(),.:]|(?<=\w)\.(?=\w))+))")
class FOLParseError(Exception): pass
class TooBig(Exception): pass

def fol_tokens(s):
    for pat, rep in _ASCII: s = re.sub(pat, rep, s)
    pos, out = 0, []
    s = s.strip()
    while pos < len(s):
        m = _FOL_TOK.match(s, pos)
        if not m or m.end() == pos: raise FOLParseError("bad char at %d: %r" % (pos, s[pos:pos + 10]))
        out.append(m.group(1) or m.group(2).lower()); pos = m.end()   # symbols are case-insensitive: gold formulas spell the same constant/predicate with different capitalisation
    return out

class _P:
    def __init__(self, toks, scope="tight", tolerant=False): self.t, self.i, self.bound, self.scope, self.tolerant = toks, 0, [], scope, tolerant
    def peek(self): return self.t[self.i] if self.i < len(self.t) else None
    def eat(self, x=None):
        v = self.peek()
        if v is None or (x is not None and v != x): raise FOLParseError(f"expected {x} got {v}")
        self.i += 1; return v
    def parse(self):
        f = self.iff()
        while self.tolerant and self.peek() == ")": self.eat()                 # stray closing parentheses in the gold annotation
        if self.peek() is not None: raise FOLParseError("trailing " + str(self.peek()))
        return f
    def iff(self):
        a = self.imp()
        while self.peek() == "↔": self.eat(); a = ("iff", a, self.imp())
        return a
    def imp(self):
        a = self.disj()
        if self.peek() == "→": self.eat(); return ("imp", a, self.imp())      # right associative
        return a
    def disj(self):
        a = self.xor()
        while self.peek() == "∨": self.eat(); a = ("or", a, self.xor())
        return a
    def xor(self):
        a = self.conj()
        while self.peek() == "⊕": self.eat(); a = ("xor", a, self.conj())
        return a
    def conj(self):
        a = self.unary()
        while self.peek() == "∧": self.eat(); a = ("and", a, self.unary())
        return a
    def unary(self):
        t = self.peek()
        if t == "¬": self.eat(); return ("not", self.unary())
        if t in ("∀", "∃"):
            self.eat(); vs = []
            while self.peek() not in (None, "(", "¬", "∀", "∃") and self.peek() not in ("∧", "∨", "→"):
                vs.append(self.eat())
                if self.peek() == ",": self.eat()
                if self.peek() in (".", ":"): self.eat(); break
                if self.peek() == "(" or self.peek() in ("¬", "∀", "∃"): break
            if not vs: raise FOLParseError("quantifier without variable")
            self.bound.extend(vs); start = self.i; body = self.unary_or_group()
            if self.scope == "max" and self.peek() in ("→", "∧", "∨", "⊕", "↔"):   # quantifier scope reaches as far right as possible
                self.i = start; body = self.iff()
            for _ in vs: self.bound.pop()
            for v in reversed(vs): body = ("forall" if t == "∀" else "exists", v, body)
            return body
        if t == "(":
            save = self.i; self.eat()
            try:
                f = self.iff(); self.eat(")")
                if self.peek() in ("=", "≠"):                              # parenthesised term followed by = : unlikely; fall through
                    raise FOLParseError("term")
                return f
            except FOLParseError:
                self.i = save
        return self.atom()
    def unary_or_group(self):
        # quantifier body: maximal scope when parenthesised group is followed by nothing binding tighter than the group itself
        return self.unary()
    def term(self):
        name = self.eat()
        if name in ("(", ")", ",", "∧", "∨", "¬", "→", "↔", "⊕", "=", "≠"): raise FOLParseError("term expected, got " + name)
        if self.peek() == "(":
            self.eat("("); args = [self.term()]
            while self.peek() == ",": self.eat(); args.append(self.term())
            self.eat(")"); return ("fn", name, tuple(args))
        return ("var", name) if name in self.bound else ("sym", name)
    def atom(self):
        name = self.eat()
        if name in ("(", ")", ",", "∧", "∨", "→", "↔", "⊕", "=", "≠"): raise FOLParseError("atom expected, got " + name)
        if self.peek() == "(":
            self.eat("("); args = [self.term()]
            while self.peek() == ",": self.eat(); args.append(self.term())
            self.eat(")")
            a = ("atom", name, tuple(args))
            if self.peek() in ("=", "≠"): raise FOLParseError("function equality unsupported")
            return a
        if self.peek() in ("=", "≠"):                                      # equality between simple terms
            op = self.eat(); lhs = ("var", name) if name in self.bound else ("sym", name); rhs = self.term()
            a = ("atom", "=", (lhs, rhs)); return a if op == "=" else ("not", a)
        return ("atom", name, ())                                          # propositional symbol

FOL_SCOPE = "tight"      # "tight": (∀x A) → B ;  "max": ∀x (A → B).  Chosen on the public dev data (see experiments).
def parse_fol(s, scope=None):
    toks = fol_tokens(s)
    if not toks: raise FOLParseError("empty")
    scope = scope or FOL_SCOPE
    try: f = _P(list(toks), scope).parse()
    except FOLParseError:
        fixed, depth = [], 0                                                       # tolerant repair of unbalanced gold formulas
        for t in toks:
            if t == "(": depth += 1
            elif t == ")":
                if depth == 0: continue                                            # drop an unmatched ')'
                depth -= 1
            fixed.append(t)
        f = _P(fixed + [")"] * depth, scope, tolerant=True).parse()               # append the missing ')'
    free = sorted({t[1] for t in _free_syms(f) if len(t[1]) == 1 and t[1].isalpha() and t[1].islower() and t[1] in "uvwxyzt"})
    for v in reversed(free): f = ("forall", v, _bind(f, v))                # free variables are implicitly universal (Prover9 convention)
    return f
def _free_syms(f):
    if f[0] == "atom":
        out = []
        def go(t):
            if t[0] == "sym": out.append(t)
            elif t[0] == "fn": [go(x) for x in t[2]]
        [go(a) for a in f[2]]; return out
    if f[0] in ("forall", "exists"): return _free_syms(f[2])
    return [x for g in f[1:] for x in _free_syms(g)]
def _bind(f, v):
    def bt(t):
        if t[0] == "sym" and t[1] == v: return ("var", v)
        if t[0] == "fn": return ("fn", t[1], tuple(bt(x) for x in t[2]))
        return t
    if f[0] == "atom": return ("atom", f[1], tuple(bt(a) for a in f[2]))
    if f[0] in ("forall", "exists"): return (f[0], f[1], _bind(f[2], v)) if f[1] != v else f
    return (f[0],) + tuple(_bind(g, v) for g in f[1:])

def nnf(f, neg=False):
    k = f[0]
    if k == "atom": return ("lit", not neg, f[1], f[2])
    if k == "not": return nnf(f[1], not neg)
    if k == "and": return (("or" if neg else "and"), nnf(f[1], neg), nnf(f[2], neg))
    if k == "or": return (("and" if neg else "or"), nnf(f[1], neg), nnf(f[2], neg))
    if k == "imp": return nnf(("or", ("not", f[1]), f[2]), neg)
    if k == "iff": return nnf(("and", ("imp", f[1], f[2]), ("imp", f[2], f[1])), neg)
    if k == "xor": return nnf(("and", ("or", f[1], f[2]), ("not", ("and", f[1], f[2]))), neg)
    if k in ("forall", "exists"):
        q = ("exists" if k == "forall" else "forall") if neg else k
        return (q, f[1], nnf(f[2], neg))
    raise FOLParseError("bad node " + k)

_sk = itertools.count(1); _vn = itertools.count(1)
def _subst_term(t, env):
    if t[0] == "var": return env.get(t[1], "?" + t[1])
    if t[0] == "sym": return t[1]
    return (t[1],) + tuple(_subst_term(x, env) for x in t[2])
def skolemize(f, env, univ):
    k = f[0]
    if k == "lit": return ("lit", f[1], f[2], tuple(_subst_term(a, env) for a in f[3]))
    if k in ("and", "or"): return (k, skolemize(f[1], env, univ), skolemize(f[2], env, univ))
    if k == "forall":
        nv = "?v%d" % next(_vn); return skolemize(f[2], {**env, f[1]: nv}, univ + [nv])
    if k == "exists":
        sk = "sk%d" % next(_sk); term = (sk,) + tuple(univ) if univ else sk
        return skolemize(f[2], {**env, f[1]: term}, univ)
    raise FOLParseError("bad nnf node")
def to_cnf(f, cap=1500):
    k = f[0]
    if k == "lit": return [[(f[1], f[2], f[3])]]
    if k == "and": return to_cnf(f[1], cap) + to_cnf(f[2], cap)
    if k == "or":
        a, b = to_cnf(f[1], cap), to_cnf(f[2], cap)
        if len(a) * len(b) > cap: raise TooBig()
        return [x + y for x in a for y in b]

# ---- terms: variable = "?name", constant = "name", function term = (name, arg, ...) ----
isvar = lambda t: isinstance(t, str) and t.startswith("?")
def walk(t, s):
    while isvar(t) and t in s: t = s[t]
    return t
def occurs(v, t, s):
    t = walk(t, s)
    if t == v: return True
    return isinstance(t, tuple) and any(occurs(v, x, s) for x in t[1:])
def unify_t(a, b, s):
    a, b = walk(a, s), walk(b, s)
    if a == b: return s
    if isvar(a): return None if occurs(a, b, s) else {**s, a: b}
    if isvar(b): return None if occurs(b, a, s) else {**s, b: a}
    if isinstance(a, tuple) and isinstance(b, tuple) and a[0] == b[0] and len(a) == len(b):
        for x, y in zip(a[1:], b[1:]):
            s = unify_t(x, y, s)
            if s is None: return None
        return s
    return None
def resolve_t(t, s):
    t = walk(t, s)
    return (t[0],) + tuple(resolve_t(x, s) for x in t[1:]) if isinstance(t, tuple) else t
def tdepth(t): return 1 + max((tdepth(x) for x in t[1:]), default=0) if isinstance(t, tuple) else 0
def rename_clause(c, tag):
    def rt(t): return (t[0],) + tuple(rt(x) for x in t[1:]) if isinstance(t, tuple) else (t + tag if isvar(t) else t)
    return frozenset((sg, p, tuple(rt(a) for a in args)) for sg, p, args in c)
def apply_clause(c, s): return frozenset((sg, p, tuple(resolve_t(a, s) for a in args)) for sg, p, args in c)
def canon(c):
    lits = sorted(c, key=lambda l: (l[1], l[0], len(l[2])))
    mp = {}
    def ct(t):
        if isvar(t): return mp.setdefault(t, "?%d" % len(mp))
        return (t[0],) + tuple(ct(x) for x in t[1:]) if isinstance(t, tuple) else t
    return frozenset((sg, p, tuple(ct(a) for a in args)) for sg, p, args in lits)
def is_taut(c): return any((not sg, p, args) in c for sg, p, args in c)
def match_t(p, t, s):                                   # one-way matching: variables of p only
    if isvar(p):
        if p in s: return s if s[p] == t else None
        return {**s, p: t}
    if isinstance(p, tuple):
        if not (isinstance(t, tuple) and p[0] == t[0] and len(p) == len(t)): return None
        for x, y in zip(p[1:], t[1:]):
            s = match_t(x, y, s)
            if s is None: return None
        return s
    return s if p == t else None

def factors(c):
    out = []
    lits = list(c)
    for i in range(len(lits)):
        for j in range(i + 1, len(lits)):
            a, b = lits[i], lits[j]
            if a[0] == b[0] and a[1] == b[1] and len(a[2]) == len(b[2]):
                s = {}
                for x, y in zip(a[2], b[2]):
                    s = unify_t(x, y, s)
                    if s is None: break
                if s is not None: out.append(apply_clause(c - {b}, s))
    return out
def binary_resolvents(c1, c2, tag):
    c2r = rename_clause(c2, tag); out = []
    for l1 in c1:
        for l2 in c2r:
            if l1[0] != l2[0] and l1[1] == l2[1] and len(l1[2]) == len(l2[2]):
                s = {}
                for x, y in zip(l1[2], l2[2]):
                    s = unify_t(x, y, s)
                    if s is None: break
                if s is not None: out.append(apply_clause((c1 - {l1}) | (c2r - {l2}), s))
    return out

STRATEGIES = ("bfs", "dfs", "ucs", "astar")
def resolution_refute(usable, sos, strategy="astar", max_clauses=4000, max_lits=10, max_depth=6, time_limit=2.0):
    """Returns 'proved' (empty clause derived), 'saturated' (no more resolvents: satisfiable) or 'limit'."""
    t_end = time.perf_counter() + time_limit
    seen, units, processed, heap, cnt = set(), [], [], [], itertools.count()
    def subsumed(c):
        for u in units:
            sg, p, args = u
            for l in c:
                if l[0] == sg and l[1] == p and len(l[2]) == len(args):
                    s = {}
                    for x, y in zip(args, l[2]):
                        s = match_t(x, y, s)
                        if s is None: break
                    if s is not None: return True
        return False
    def push(c, gen):
        if is_taut(c) or len(c) > max_lits or any(tdepth(a) > max_depth for _, _, args in c for a in args): return
        k = canon(c)
        if k in seen or subsumed(k): return
        seen.add(k)
        if len(k) == 1: units.append(next(iter(k)))
        w = len(k)
        key = {"bfs": (gen, next(cnt)), "dfs": (-gen, -next(cnt)), "ucs": (w, gen, next(cnt)), "astar": (gen + w, w, next(cnt))}[strategy]
        heapq.heappush(heap, (key, next(cnt), k, gen))
    for c in usable:
        k = canon(frozenset(c))
        if not k: return "proved"
        seen.add(k); processed.append(k)
        if len(k) == 1: units.append(next(iter(k)))
    for c in sos:
        k = frozenset(c)
        if not k: return "proved"
        push(k, 0)
    n = 0
    while heap:
        if time.perf_counter() > t_end or len(seen) > max_clauses: return "limit"
        _, _, g, gen = heapq.heappop(heap)
        cands = [g] + factors(g)
        for gc in cands:
            if gc is not g: push(gc, gen + 1)
        for other in processed + [g]:
            n += 1
            for r in binary_resolvents(g, other, "_%d" % n):
                if not r: return "proved"
                push(r, gen + 1)
                if time.perf_counter() > t_end: return "limit"
        processed.append(g)
    return "saturated"

def fol_to_clauses(f):
    return [frozenset(c) for c in to_cnf(skolemize(nnf(f), {}, []))]

def fol_entails(premises_text, conclusion_text, strategy="astar", time_limit=2.0, want_negation=False):
    """Returns (verdict, info) with verdict in {'proved','saturated','limit'}; want_negation=True tests premises ⊨ ¬conclusion."""
    plines = [l.strip() for l in re.split(r"\n", str(premises_text)) if l.strip()]
    prem = []
    for l in plines: prem += fol_to_clauses(parse_fol(l))
    cf = parse_fol(str(conclusion_text).strip().split("\n")[0])
    neg = ("not", cf) if not want_negation else cf
    # free variables of the conclusion are universal in the conclusion, i.e. existential after negation: parse_fol already closed them universally
    sos = fol_to_clauses(neg)
    return resolution_refute(prem, sos, strategy, time_limit=time_limit), (len(prem), len(sos))

_FOL_MARK = re.compile(r"[∀∃∧∨¬→↔⊕]|\bforall\b|\bexists\b")
def looks_like_fol(text): return bool(_FOL_MARK.search(str(text))) and "(" in str(text)

## 3. End-to-end `answer(context, query)`

In [6]:
def build_kb(context, semantics=None):
    semantics = semantics or SEMANTICS
    fl, rules = parse_context(context)
    domain = {c for l in fl for c in l.args} | {c for r in rules for l in list(r.body) + [r.head] for c in l.args if not is_var(c)}
    if semantics == "owa":
        tr = lambda l: Lit(False, "~" + l.pred, l.args) if l.neg else l      # strong negation as separate atom ~p
    else:
        # closed world: "not p" in bodies/queries = negation as failure. Explicit negative facts and negative rule
        # heads are redundant (a consistent KB never derives p when it asserts not-p), so they are dropped.
        fl = [l for l in fl if not l.neg]; rules = [r for r in rules if not r.head.neg]
        tr = lambda l: l
    facts = {(l.pred, l.args) for l in map(tr, fl)}
    rules = [Rule(tuple(tr(b) for b in r.body), tr(r.head), r.text) for r in rules]
    return facts, rules, tr, domain

def answer(context, query, engine=None, trace=False, semantics=None):
    engine = engine or ENGINE
    if looks_like_fol(context) and not is_template_context(context):
        try: v, (np_, ns_) = fol_entails(context, query, FOL_STRATEGY, FOL_TIME_LIMIT)
        except Exception as ex_: v, np_, ns_ = "error:" + type(ex_).__name__, 0, 0
        res = (v == "proved")
        if not trace: return res
        return res, [f"FOL: {np_} premise clauses + {ns_} clause(s) from the negated conclusion; resolution/{FOL_STRATEGY} -> {v}"
                     + (": empty clause derived, so the conclusion is entailed" if res else ": no refutation, so not entailed")]
    cx = str(context).rstrip(); full = (cx + ("" if cx[-1:] in ".?!" or not cx else ".") + " " if cx else "") + str(query)
    if looks_like_counterbench(full):
        r = answer_counterbench(full, trace)
        if r is not None: return r
    facts, rules, tr, domain = build_kb(context, semantics)
    ql = parse_query(str(query).lower() if is_template_context(context) else query)
    domain |= {c for l in ql for c in l.args if not is_var(c)}
    q = [tr(l) for l in ql]
    if not q: return (False, ["(could not parse query)"]) if trace else False
    facts, rules = prune(facts, rules, [l.pred for l in q])
    if engine == "bc":
        res = BackwardChainer(facts, rules).prove(q)
        return (res, ["(backward chaining; no trace)"]) if trace else res
    fc = ForwardChainer(facts, rules, domain)
    goal = (q[0].pred, q[0].args) if len(q) == 1 and not q[0].neg else None
    fc.run(goal)
    res = all(fc.holds(l) for l in q)
    if not trace: return res
    lines = []
    for l in q:
        if l.neg: lines.append(f"not {fmt(l.pred,l.args)}: {'holds by negation as failure' if fc.holds(l) else 'FAILS (it is derivable)'}")
        elif fc.holds(l): lines += fc.explain((l.pred, l.args))
        else: lines.append(f"{fmt(l.pred,l.args)}: not derivable -> False")
    return res, lines

## 4. Demo & sanity tests (assignment example + typical RuleTaker patterns)

In [7]:
demo_ctx = """Bob is young.
Bob is kind.
If someone is young, then they are smart.
If someone is smart and kind, then they are successful."""
r, tr_ = answer(demo_ctx, "Bob is successful.", trace=True)
print("Answer:", r); print("\n".join(tr_))

CB_DEMO = ("We know that Nuv causes Splee, Blen and Druk, not Druk causes Plog, Plog causes Skrim, Skrim or Druk causes Zimb, Zimb causes Yurd, and Yurd causes Wrox. "
           "Would Wrox occur if not Nuv and not Splee?")
ok, lines = answer("", CB_DEMO, trace=True); print("\nCounterBench demo ->", ok); print("\n".join(lines))

TESTS = [
 (demo_ctx, "Bob is successful.", True),
 (demo_ctx, "Bob is not kind.", False),
 ("", CB_DEMO, True),
 ("The cat is big. The cat chases the dog. If something chases the dog then it is red. Red things are nice. If something is nice and not big then it is cute.", "The cat is nice.", True),
 ("The cat is big. Red things are nice. Big, red things are rough. The dog is red.", "The dog is rough.", False),
 ("The cat is big. The dog is red. If something is not big then it is small.", "The dog is small.", True),   # CWA/NAF
 ("The cat is not big. The dog is big. All big things are red.", "The cat is red.", False),
 ("The cat is not big. The dog is big. If something is not big then it is red.", "The cat is red.", True),
 ("Anne is nice. Nice people are kind. Kind people are happy. If someone is happy then they like the bear. Everything is old.", "Anne likes the bear.", True),
 ("The bald eagle needs the dog. If the bald eagle needs the dog then the dog is big. If something is big and it does not need the bear then it is green.", "The dog is green.", True),
]
for ctx, q, gold in TESTS:
    for eng in ("fc", "bc"):
        assert answer(ctx, q, eng) == gold, (eng, ctx, q)
print("All sanity tests passed for both engines.")

Answer: True
successful(bob)   [by rule: If someone is smart and kind, then they are successful.]
  smart(bob)   [by rule: If someone is young, then they are smart.]
    young(bob)   [given fact]
  kind(bob)   [given fact]

CounterBench demo -> True
exogenous variables: ['Nuv'], priors: uniform 0.5
evidence: []
action: ['do(Nuv=0)', 'do(Splee=0)']
posterior P(Wrox | evidence, do) = 1.000
occurs(Wrox)   [by rule: Yurd causes Wrox]
  occurs(Yurd)   [by rule: Zimb causes Yurd]
    occurs(Zimb)   [by rule: Skrim or Druk causes Zimb]
      occurs(Skrim)   [by rule: Plog causes Skrim]
        occurs(Plog)   [by rule: not Druk causes Plog]
          not occurs(Druk)   [negation as failure: not derivable]
All sanity tests passed for both engines.


## 5. Data loading
The notebook first looks for local files in `DATA_DIR` (layout in `DEV_FILES`) and otherwise **downloads the public datasets from Hugging Face** (`jise/ruletaker`, `yale-nlp/FOLIO`, `CounterBench/CounterBench`; needs `pip install datasets huggingface_hub` and internet). Column names are detected automatically, so the hidden test file may use any of them:
* **RuleTaker** (`context` = "Facts:/Rules:" text, `statement`, `flag`), **FOLIO** (`premises-FOL`, `conclusion-FOL`, `label`: logic is given directly), **CounterBench** (`given_info`, `question`, `answer` yes/no).
* For the binary task every label other than true/entailment/yes (False, Uncertain, not entailment, no) is **False**.
* The router in `answer()` looks only at the *format* of the input (FOL symbols → resolution prover; "causes…/Would … occur" → causal-model solver; otherwise rule base).

In [8]:
import pandas as pd, random
COLS = dict(ctx=["premises-fol", "premises_fol", "premises_logic", "context", "given_info", "theory", "premises", "knowledge_base", "kb", "story", "input", "text", "prompt"],
            q=["conclusion-fol", "conclusion_fol", "conclusion_logic", "statement", "question", "query", "conclusion", "hypothesis"],
            y=["label", "flag", "answer", "gold", "target"],
            id=["id", "example_id", "question_id", "idx", "qid"],
            cfg=["config", "depth", "type", "category", "level"])
def pick(cols, cands):
    low = {str(c).lower(): c for c in cols}
    for c in cands:
        if c.lower() in low: return low[c.lower()]
    return None
def to_bool(v):
    if v is None or (isinstance(v, float) and v != v): return None
    return str(v).strip().lower() in {"true", "entailment", "entailed", "1", "yes", "t", "y"}
def normalise(df, name=""):
    c, q, y, i, g = (pick(df.columns, COLS[k]) for k in ("ctx", "q", "y", "id", "cfg"))
    if q is None and c is None: raise ValueError(f"cannot find text columns in {list(df.columns)}")
    out = []
    for k in range(len(df)):
        ctxv = df[c].iloc[k] if c is not None and c != q else ""
        if isinstance(ctxv, (list, tuple)) or hasattr(ctxv, "tolist"): ctxv = " ".join(map(str, list(ctxv)))
        out.append(dict(id=(df[i].iloc[k] if i else k), context=ctxv, query=(df[q].iloc[k] if q is not None else ""),
                        label=(to_bool(df[y].iloc[k]) if y else None), config=(str(df[g].iloc[k]) if g else name)))
    return out
def read_any(path):
    ext = os.path.splitext(path)[1].lower()
    return {".csv": pd.read_csv, ".jsonl": lambda p: pd.read_json(p, lines=True), ".json": pd.read_json,
            ".parquet": pd.read_parquet}[ext](path)
def stratified_sample(exs, n, seed=SEED):
    if not n or len(exs) <= n: return exs
    rng = random.Random(seed); by = defaultdict(list)
    for e in exs: by[e["config"]].append(e)
    out = []
    for g, lst in by.items(): out += rng.sample(lst, max(1, round(n * len(lst) / len(exs))))
    return out[:n] if len(out) > n else out

# Hugging Face sources named in the assignment. Used automatically when a local file is missing.
HF_SPLITS = {"RuleTaker-train": ("jise/ruletaker", "train"), "RuleTaker-test": ("jise/ruletaker", "test"), "RuleTaker-ood": ("jise/ruletaker", "ood_test"),
             "FOLIO-train": ("yale-nlp/FOLIO", "train"), "FOLIO-validation": ("yale-nlp/FOLIO", "validation")}
HF_CB_REPO = "CounterBench/CounterBench"     # its two data files are fetched one by one (the repo mixes two schemas, so load_dataset() on it fails)
HF_CB_FILES = {"CounterBench-alpha": "data_balanced_alpha_V1.json", "CounterBench-backdoor": "data_balanced_backdoor_V2.json"}
_hf_cache = {}
def load_from_hf(name):
    if name in HF_CB_FILES:
        from huggingface_hub import hf_hub_download
        return pd.read_json(hf_hub_download(repo_id=HF_CB_REPO, filename=HF_CB_FILES[name], repo_type="dataset"))
    repo, split = HF_SPLITS[name]
    if repo not in _hf_cache:
        from datasets import load_dataset
        _hf_cache[repo] = load_dataset(repo)
    ds = _hf_cache[repo]
    if split not in ds: split = {"validation": "dev", "ood_test": "ood"}.get(split, split)
    if split not in ds: raise KeyError(f"split not found in {repo}: have {list(ds.keys())}")
    return ds[split].to_pandas()

dev_sets, dev_source = {}, {}
for name, rel in DEV_FILES.items():
    path = os.path.join(DATA_DIR, rel)
    try:
        if os.path.exists(path): df, src = read_any(path), path
        else: df, src = load_from_hf(name), "Hugging Face"
        dev_sets[name] = normalise(df, name); dev_source[name] = src
    except Exception as ex:
        if "gated" in str(ex).lower() or type(ex).__name__ in ("DatasetNotFoundError", "GatedRepoError"):
            print(f"[{name}] ACCESS REQUIRED: {HF_SPLITS.get(name, ('?',))[0]} is a gated Hugging Face dataset. (1) Log in at huggingface.co and open "
                  f"https://huggingface.co/datasets/yale-nlp/FOLIO, click the button to accept the terms; (2) create a Read token under Settings > Access Tokens; "
                  f"(3) run `hf auth login` (or `huggingface-cli login`) in a terminal and paste the token; then re-run. Alternatively place the file at {path}.")
        else:
            print(f"[{name}] could not be loaded ({type(ex).__name__}: {str(ex)[:150]}). Fix: pip install datasets huggingface_hub (and check your internet), "
                  f"or place the file at {path}")
if not dev_sets: print("WARNING: no public dataset could be loaded, so the experiments below are skipped; the reasoning engines themselves still work.")
for name, exs in dev_sets.items():
    print(f"--- {name}: {len(exs)} records, label balance {sum(bool(e['label']) for e in exs)}/{len(exs)} True")
    e = exs[0]; print("    ctx=%r\n    q=%r  label=%s config=%s" % (str(e["context"])[:110], str(e["query"])[:90], e["label"], e["config"]))

--- RuleTaker-train: 1000 records, label balance 594/1000 True
    ctx='Facts:\nThe wall dances gracefully.\nThe river dances gracefully.\nThe river becomes famous.\nRules:\nAssuming the '
    q='The sandwich dances gracefully.'  label=False config=2
--- RuleTaker-test: 250 records, label balance 156/250 True
    ctx='Facts:\nLiam volunteers.\nThe shop sings beautifully.\nLiam volunteers.\nRules:\nWhen liam volunteers, we can concl'
    q='The book volunteers.'  label=False config=3
--- RuleTaker-ood: 375 records, label balance 131/375 True
    ctx='Facts:\nThe pumpkin solves puzzles.\nSarah is interesting.\nSarah solves puzzles.\nRules:\nBased on sarah solves pu'
    q='The teacher plays instruments.'  label=True config=4
--- FOLIO-train: 1001 records, label balance 388/1001 True
    ctx='∀x (DrinkRegularly(x, coffee) → IsDependentOn(x, caffeine))\n∀x (DrinkRegularly(x, coffee)  ∨ (¬WantToBeAddicte'
    q='¬WantToBeAddictedTo(rina, caffeine) ∨ (¬AwareThatDrug(rina, caffeine))'  label=

## 6. Parser diagnostics, engine validation and public-data experiments
Per dataset: size, accuracy and time of every applicable engine, **parse failures** (sentences/formulas the parser could not understand), forward-vs-backward
disagreements, per-config accuracy and example errors. The number printed as `time` is the total inference time over the whole evaluation set.

In [9]:
def kind_of(ex):
    c = str(ex["context"])
    if looks_like_fol(c) and not is_template_context(c): return "fol"
    if looks_like_counterbench(c + " " + str(ex["query"])): return "scm"
    return "rules"
ENGINES = {"rules": ("fc", "bc"), "fol": ("auto",), "scm": ("auto",)}

def diagnose(context, query):
    kind = kind_of(dict(context=context, query=query))
    if kind == "fol":
        bad_s = []
        for l in [x for x in str(context).split("\n") if x.strip()] + [str(query)]:
            try: fol_to_clauses(parse_fol(l))
            except Exception: bad_s.append(l[:120])
        return dict(n=len([x for x in str(context).split("\n") if x.strip()]) + 1, bad=len(bad_s), qbad=0, bad_sents=bad_s)
    if kind == "scm":
        cx = str(context).rstrip(); full = (cx + ("" if cx[-1:] in ".?!" or not cx else ".") + " " if cx else "") + str(query)
        ok = parse_counterbench(full) is not None
        return dict(n=1, bad=0 if ok else 1, qbad=0, bad_sents=[] if ok else [full[:120]])
    if is_template_context(context):
        f, r, failed = parse_template_context(context)
        return dict(n=len(f) + len(r) + len(failed), bad=len(failed), qbad=0, bad_sents=failed)
    n = bad = 0; bad_s = []
    for s_ in split_sentences(context):
        r = parse_sentence(s_); n += 1
        if r is None: continue
        lits = r[1] if r[0] == "fact" else [l for ru in r[1] for l in list(ru.body) + [ru.head]]
        if any(l.pred.startswith("prop_") for l in lits): bad += 1; bad_s.append(s_)
    ql = parse_query(query)
    return dict(n=n, bad=bad, qbad=int(any(l.pred.startswith("prop_") for l in ql) or not ql), bad_sents=bad_s)

def predict(ex, engine=None):
    try: return bool(answer(ex["context"], ex["query"], None if engine == "auto" else engine))
    except Exception: return False       # safe default on parser/engine failure

RESULTS = {}
def run_experiment(name, exs, n_max=MAX_EVAL_PER_DATASET):
    exs = [e for e in exs if e["label"] is not None]
    pool = len(exs); exs = stratified_sample(exs, n_max)
    kind = kind_of(exs[0]); res = dict(source=dev_source.get(name, "?"), pool=pool, n=len(exs), kind=kind, engines={})
    preds = {}
    for eng in ENGINES[kind]:
        t0 = time.perf_counter(); preds[eng] = [predict(e, eng) for e in exs]; dt = time.perf_counter() - t0
        acc = sum(p == e["label"] for p, e in zip(preds[eng], exs)) / max(1, len(exs))
        res["engines"][eng] = dict(accuracy=100 * acc, total_time_s=dt, ms_per_query=1000 * dt / max(1, len(exs)))
    main = ENGINES[kind][0]
    dis = [i for i in range(len(exs)) if len({preds[e][i] for e in preds}) > 1]
    res["fc_bc_disagree"] = len(dis)
    dg = [diagnose(e["context"], e["query"]) for e in exs]
    res["parse"] = dict(sentences=sum(d["n"] for d in dg), unparsed_sentences=sum(d["bad"] for d in dg),
                        examples_with_unparsed=sum(1 for d in dg if d["bad"] or d["qbad"]), unparsed_queries=sum(d["qbad"] for d in dg),
                        samples=[b for d in dg for b in d["bad_sents"]][:8])
    res["errors"] = [dict(context=str(e["context"])[:600], query=e["query"], gold=e["label"], pred=p) for e, p in zip(exs, preds[main]) if p != e["label"]][:5]
    per = defaultdict(lambda: [0, 0])
    for e, p in zip(exs, preds[main]): per[e["config"]][0] += (p == e["label"]); per[e["config"]][1] += 1
    res["per_config"] = {k: dict(n=v[1], accuracy=100 * v[0] / v[1]) for k, v in sorted(per.items())}
    RESULTS[name] = res
    es = " | ".join(f"{k} acc={v['accuracy']:.2f}% {v['total_time_s']:.2f}s" for k, v in res["engines"].items())
    print(f"[{name}] ({kind}) n={res['n']} | {es} | engine disagreements: {len(dis)} | unparsed {res['parse']['unparsed_sentences']}/{res['parse']['sentences']} "
          f"(examples affected {res['parse']['examples_with_unparsed']})")
    for k, v in res["per_config"].items():
        if len(res["per_config"]) > 1: print(f"      {k:>14}: n={v['n']:5d}  acc={v['accuracy']:.2f}%")
    return res

for name, exs in dev_sets.items(): run_experiment(name, exs)
tot_n = sum(r["n"] for r in RESULTS.values()); tot_ok = sum(r["n"] * r["engines"][ENGINES[r["kind"]][0]]["accuracy"] / 100 for r in RESULTS.values())
tot_t = sum(r["engines"][ENGINES[r["kind"]][0]]["total_time_s"] for r in RESULTS.values())
RESULTS_SUMMARY = dict(n=tot_n, accuracy=100 * tot_ok / max(1, tot_n), time_s=tot_t); print(f"OVERALL (main engine per dataset): {tot_ok:.0f}/{tot_n} = {RESULTS_SUMMARY['accuracy']:.2f}%  total inference time {tot_t:.1f}s")
json.dump(RESULTS, open("results.json", "w"), indent=1, default=str)

[RuleTaker-train] (rules) n=1000 | fc acc=98.30% 0.27s | bc acc=98.30% 0.31s | engine disagreements: 0 | unparsed 0/10267 (examples affected 0)
                   1: n=  196  acc=100.00%
                   2: n=  406  acc=96.80%
                   3: n=  398  acc=98.99%
[RuleTaker-test] (rules) n=250 | fc acc=97.60% 0.07s | bc acc=97.60% 0.07s | engine disagreements: 0 | unparsed 0/2591 (examples affected 0)
                   1: n=   54  acc=100.00%
                   2: n=   94  acc=94.68%
                   3: n=  102  acc=99.02%


[RuleTaker-ood] (rules) n=375 | fc acc=99.47% 0.09s | bc acc=99.47% 0.08s | engine disagreements: 0 | unparsed 0/3663 (examples affected 0)
                   4: n=  250  acc=100.00%
                   5: n=  125  acc=98.40%


[FOLIO-train] (fol) n=1001 | auto acc=80.32% 74.73s | engine disagreements: 0 | unparsed 29/6286 (examples affected 28)


[FOLIO-validation] (fol) n=203 | auto acc=84.24% 31.45s | engine disagreements: 0 | unparsed 18/1307 (examples affected 9)


[CounterBench-alpha] (scm) n=1000 | auto acc=96.00% 0.38s | engine disagreements: 0 | unparsed 0/1000 (examples affected 0)
               basic: n=  250  acc=100.00%
         conditional: n=  250  acc=92.00%
               joint: n=  250  acc=96.00%
              nested: n=  250  acc=96.00%
[CounterBench-backdoor] (scm) n=200 | auto acc=90.00% 0.07s | engine disagreements: 0 | unparsed 0/200 (examples affected 0)
OVERALL (main engine per dataset): 3715/4029 = 92.21%  total inference time 107.1s


### Proof-search strategy comparison on FOLIO (resolution given-clause selection: BFS / DFS / UCS / A*)

In [10]:
if "FOLIO-validation" in dev_sets:
    FOLEX = dev_sets["FOLIO-validation"]; keep = FOL_STRATEGY; RESULTS["fol_strategies"] = {}
    for st in STRATEGIES:
        FOL_STRATEGY = st; t0 = time.perf_counter(); p = [predict(e) for e in FOLEX]; dt = time.perf_counter() - t0
        acc = 100 * sum(a == e["label"] for a, e in zip(p, FOLEX)) / len(FOLEX)
        RESULTS["fol_strategies"][st] = dict(accuracy=acc, time_s=dt); print(f"FOLIO-validation  resolution/{st:5s}: accuracy {acc:.1f}%  time {dt:.1f}s")
    FOL_STRATEGY = keep

FOLIO-validation  resolution/bfs  : accuracy 83.3%  time 40.1s


FOLIO-validation  resolution/dfs  : accuracy 84.7%  time 57.3s


FOLIO-validation  resolution/ucs  : accuracy 84.7%  time 30.6s


FOLIO-validation  resolution/astar: accuracy 84.2%  time 31.6s


### Engine validation by differential testing
Random RuleTaker-style knowledge bases (negation, relations, recursion, "All/Every" rules) are answered by both engines.
Closed-world negation is only well defined for *stratifiable* programs, so disagreements are split by stratifiability.

In [11]:
ADJ=["big","red","kind","nice","rough","cold","round","young","smart"]; ENT=["Anne","Bob","Dave","the cat","the dog","the bald eagle"]; VERB=["likes","chases","needs","visits"]
def rand_kb(rng):
    S=[]
    for e in rng.sample(ENT,rng.randint(2,4)):
        for a in rng.sample(ADJ,rng.randint(1,3)): S.append(f"{e[0].upper()+e[1:]} is {'not ' if rng.random()<.15 else ''}{a}.")
    for _ in range(rng.randint(0,3)):
        a,b=rng.sample(ENT,2); S.append(f"{a[0].upper()+a[1:]} {rng.choice(VERB)} {b}.".replace("The ","The "))
    for _ in range(rng.randint(2,6)):
        t=rng.random(); N=lambda: 'not ' if rng.random()<.25 else ''
        if t<.3: S.append(f"If someone is {N()}{rng.choice(ADJ)} then they are {N()}{rng.choice(ADJ)}.")
        elif t<.5: S.append(f"If something is {N()}{rng.choice(ADJ)} and {N()}{rng.choice(ADJ)} then it is {rng.choice(ADJ)}.")
        elif t<.65: S.append(f"{rng.choice(ADJ).capitalize()}, {rng.choice(ADJ)} things are {N()}{rng.choice(ADJ)}.")
        elif t<.75: S.append(f"All {rng.choice(ADJ)} people are {rng.choice(ADJ)}.")
        elif t<.88: S.append(f"If something {rng.choice(VERB)[:-1]+'s'} the {rng.choice(['cat','dog'])} then it is {rng.choice(ADJ)}.")
        else: S.append(f"If the cat is {rng.choice(ADJ)} then the cat is {N()}{rng.choice(ADJ)}.")
    return " ".join(S)

def stratifiable(kb, q):
    facts, rules, tr, dom = build_kb(kb, "cwa"); ql = parse_query(q)
    facts, rules = prune(facts, rules, [l.pred for l in ql]); st = stratify(rules)
    return all(st[r.head.pred] >= st[l.pred] + (1 if l.neg else 0) for r in rules for l in r.body)
rng = random.Random(1); N_DIFF = 4000; stat = dict(cases=0, stratifiable=0, disagree_total=0, disagree_stratifiable=0); ex_dis = []
for _ in range(N_DIFF):
    kb = rand_kb(rng); e = rng.choice(ENT); q = f"{e[0].upper()+e[1:]} is {'not ' if rng.random()<.4 else ''}{rng.choice(ADJ)}."
    a, b = answer(kb, q, "fc", semantics="cwa"), answer(kb, q, "bc", semantics="cwa"); st_ = stratifiable(kb, q)
    stat["cases"] += 1; stat["stratifiable"] += st_
    if a != b:
        stat["disagree_total"] += 1; stat["disagree_stratifiable"] += st_
        if len(ex_dis) < 2: ex_dis.append((kb, q, a, b))
print(stat)
RESULTS["difftest"] = stat

{'cases': 4000, 'stratifiable': 3918, 'disagree_total': 16, 'disagree_stratifiable': 0}


### Semantics ablation (closed-world vs open-world) and example reasoning traces

In [12]:
if "RuleTaker-test" in dev_sets:
    ex_ = dev_sets["RuleTaker-test"]
    for sem in ("cwa", "owa"):
        acc = sum(bool(answer(e["context"], e["query"], "fc", semantics=sem)) == e["label"] for e in ex_) / len(ex_)
        print(f"RuleTaker-test semantics={sem}: accuracy {100*acc:.2f}% (n={len(ex_)})")
        RESULTS.setdefault("ablation", {})[sem] = 100 * acc
    e = next((x for x in ex_ if x["label"]), ex_[0])
    ok, lines = answer(e["context"], e["query"], "fc", trace=True)
    print("\nTRACE:", e["query"], "->", ok); print("\n".join(lines))
json.dump(RESULTS, open("results.json", "w"), indent=1, default=str)

RuleTaker-test semantics=cwa: accuracy 97.60% (n=250)
RuleTaker-test semantics=owa: accuracy 97.60% (n=250)

TRACE: The pig creates art. -> True
prop_pig_creates_art   [by rule: Based on ethan is a scientist, it is evident that the pig creates art.]
  scientist(ethan)   [by rule: If jack becomes famous, then ethan is a scientist.]
    prop_jack_becomes_famous   [by rule: Assuming jack is a scientist, we can say jack becomes famous.]
      scientist(jack)   [given fact]


## 7. Private test inference → `submission.csv`

In [13]:
if os.path.exists(PRIVATE_TEST_PATH):
    test = normalise(read_any(PRIVATE_TEST_PATH), "test")
    t0 = time.perf_counter()
    preds = [predict(e) for e in test]
    dt = time.perf_counter() - t0
    pd.DataFrame({"id": [e["id"] for e in test], "label": [str(p) for p in preds]}).to_csv(SUBMISSION_PATH, index=False)
    print(f"Wrote {SUBMISSION_PATH}: {len(preds)} predictions, inference time {dt:.2f}s ({1000*dt/max(1,len(test)):.3f} ms/query); formats seen: {dict(collections.Counter(kind_of(e) for e in test))}")
    if all(e["label"] is not None for e in test):
        print("Accuracy: %.2f%%" % (100 * sum(p == e["label"] for p, e in zip(preds, test)) / len(test)))
else:
    print(f"'{PRIVATE_TEST_PATH}' not found: set PRIVATE_TEST_PATH to the test file to generate {SUBMISSION_PATH}")

'private_test.csv' not found: set PRIVATE_TEST_PATH to the test file to generate submission.csv


## 8. Generate `report.pdf` from the measured results
Every number in the report comes from `RESULTS` (also saved to `results.json`). Requires `reportlab`.

In [14]:
CB_EX1 = ("We know that Ziklo causes not Blaf, Blaf causes Trune, Trune causes not Vork, Vork or Trune causes Sline, Sline causes Frim, not Frim and Trune causes Qado, and Qado causes Lumbo. "
          "Would Lumbo occur if not Ziklo instead of Ziklo?")
def make_report(path="report.pdf"):
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib import colors
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Preformatted
    from xml.sax.saxutils import escape as esc
    ss = getSampleStyleSheet(); B = ss["BodyText"]; B.fontSize = 9.5; B.leading = 12.5
    H1 = ParagraphStyle("h1", parent=ss["Heading2"], spaceBefore=10, spaceAfter=4)
    MONO = ParagraphStyle("mono", fontName="Courier", fontSize=7.4, leading=9, backColor=colors.whitesmoke, borderPadding=3)
    P = lambda t, st=B: Paragraph(t, st)
    def tbl(rows, widths):
        t = Table(rows, hAlign="LEFT", colWidths=widths)
        t.setStyle(TableStyle([("FONTSIZE", (0, 0), (-1, -1), 8), ("GRID", (0, 0), (-1, -1), .3, colors.grey), ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey)])); return t
    def trace_block(title, ctx, q):
        ok, lines = answer(ctx, q, "fc", trace=True)
        qs = q[q.rfind("Would"):] if "Would" in q else q[-110:]
        return [P(f"<b>{esc(title)}</b>  &mdash; query: <i>{esc(qs)}</i> &rarr; <b>{ok}</b>"), Preformatted("\n".join(lines)[:1800], MONO), Spacer(1, 5)]
    main = lambda r: r["engines"][ENGINES[r["kind"]][0]]
    st = [P("<b>Classical Reasoning Agent &mdash; Report</b>", ss["Title"])]
    st += [P("1. Representation", H1), P(
        "The system chooses its representation from the <i>format</i> of the input (it never sees a dataset name). "
        "<b>Rule bases in English</b> (RuleTaker) are translated by a deterministic parser into <b>Horn clauses</b> over literals <i>Lit(neg, predicate, args)</i>. The course data is written as "
        "<i>Facts:</i> / <i>Rules:</i> with five paraphrase templates (&ldquo;When A, we can conclude B&rdquo;, &ldquo;Given that A, it follows that B&rdquo;, &ldquo;If A, then B&rdquo;, &ldquo;Based on A, it is evident that B&rdquo;, "
        "&ldquo;Assuming A, we can say B&rdquo;); every clause is lower-cased and reduced to a ground atom, so a clause means the same thing in a fact, a rule and the query. The parser also supports the classic "
        "RuleTaker forms (&ldquo;Big, red things are nice&rdquo;, &ldquo;If someone &hellip; then they &hellip;&rdquo;) with variables, relations and negation. "
        "Negation is closed-world (negation as failure); an open-world mode (strong negation as a separate atom <i>~p</i>) is kept as an ablation. Sentences that cannot be understood become opaque atoms and are counted as parse failures. "
        "<b>First-order formulas</b> (FOLIO, given directly) are parsed by a tolerant FOL parser (Unicode and ASCII connectives, quantifiers, function terms, equality), then converted to negation normal form, "
        "standardised apart, <b>Skolemised</b> and put into <b>CNF</b>; clauses are sets of literals. The parser repairs unbalanced parentheses found in the gold annotations and treats symbols case-insensitively. "
        "<b>CounterBench</b> text is translated into a structural causal model: each variable is <i>occurs(v)</i>; &ldquo;A and B together cause C&rdquo; is a conjunctive rule, &ldquo;A or B causes C&rdquo; two rules, "
        "negations use NAF (the graph is acyclic, hence stratified), &ldquo;A causes not B&rdquo; is De Morgan-negated; <i>Bern(p)</i> gives priors of exogenous variables.")]
    st += [P("2. Reasoning algorithms", H1), P(
        "<b>Forward chaining</b> (main engine for rule bases): relevance pruning on the predicate dependency graph, stratification of negation, per-stratum fixpoint, rule bodies matched by <b>unification</b> against hash-indexed facts "
        "with greedy join ordering, early exit when a positive goal is derived, provenance for proof traces. <b>Backward chaining</b>: SLD resolution with unification, standardising apart, ancestor loop check, depth bound and NAF. "
        "<b>Resolution</b> (FOL): binary resolution with factoring and <b>unification with occurs check</b>, set-of-support (premises are the usable set, the negated conclusion is the support), tautology, duplicate and unit-subsumption pruning, "
        "term-depth and clause-length bounds; the given clause is chosen by a pluggable search strategy (<b>BFS</b>: age, <b>DFS</b>: newest first, <b>UCS</b>: clause weight, <b>A*</b>: derivation depth + literal count as heuristic). "
        "premises &and; &not;conclusion unsatisfiable &rArr; True; saturation or resource limit &rArr; False. "
        "<b>Counterfactuals</b> (CounterBench) use the abduction-action-prediction scheme: enumerate exogenous worlds (search over assignments, weighted by the priors) that reproduce the evidence in the factual world, "
        "cut the incoming edges of the intervened variables, and run forward chaining in each world; the answer is Yes iff P(target | evidence, do) &gt; 1/2. No generative model is used for any prediction.")]
    st += [P("3. Experiments on the public data", H1)]
    names = [n for n in RESULTS if n not in ("ablation", "difftest", "fol_strategies")]
    rows = [["dataset", "n", "engine", "accuracy %", "total time s", "ms/query", "parse fail", "FC/BC differ"]]
    for n in names:
        r = RESULTS[n]
        for k, v in r["engines"].items():
            rows.append([n, r["n"], k, f"{v['accuracy']:.2f}", f"{v['total_time_s']:.2f}", f"{v['ms_per_query']:.2f}", f"{r['parse']['unparsed_sentences']}/{r['parse']['sentences']}", r["fc_bc_disagree"] if len(r["engines"]) > 1 else "-"])
    if names: st += [tbl(rows, [105, 32, 36, 60, 62, 48, 62, 62]), Spacer(1, 4)]
    if "RESULTS_SUMMARY" in globals(): st.append(P(f"<b>Overall</b> (main engine per dataset): {RESULTS_SUMMARY['accuracy']:.2f}% over {RESULTS_SUMMARY['n']} public examples, total inference time {RESULTS_SUMMARY['time_s']:.1f}s."))
    for n in names:
        r = RESULTS[n]
        if len(r["per_config"]) > 1:
            st.append(P(f"<b>{esc(n)}</b> by config: " + ", ".join(f"{esc(k)}: {v['accuracy']:.1f}% (n={v['n']})" for k, v in r["per_config"].items())))
    if "fol_strategies" in RESULTS:
        st.append(P("<b>Resolution strategy comparison</b> (FOLIO validation): " + "; ".join(f"{k.upper()} {v['accuracy']:.1f}% in {v['time_s']:.1f}s" for k, v in RESULTS["fol_strategies"].items()) + "."))
    if "ablation" in RESULTS: st.append(P(f"<b>Semantics ablation (RuleTaker test):</b> closed-world/NAF {RESULTS['ablation']['cwa']:.2f}% vs open-world {RESULTS['ablation']['owa']:.2f}%."))
    if "difftest" in RESULTS:
        d = RESULTS["difftest"]
        st.append(P(f"<b>Engine validation (differential testing):</b> {d['cases']} random RuleTaker-style KBs ({d['stratifiable']} stratifiable); FC and BC disagreed on {d['disagree_total']}, "
                    f"<b>{d['disagree_stratifiable']}</b> of them on stratifiable programs (the rest involve negation through recursion, which has no well-defined closed-world model)."))
    st.append(P("<b>Error analysis.</b> RuleTaker: the residual errors are items where the statement <i>is</i> derivable by a single applicable rule but the gold flag says False (label noise in the generated data); see the first stored error below. "
                "FOLIO: errors come from the gold formulas themselves (unbalanced parentheses, mismatched predicate names or argument order, constants spelled differently, missing background axioms), "
                "not from the prover; recall on True items is therefore far below precision. CounterBench: the unexplained gold labels concentrate on a few graph structures with observed evidence that contradicts the stated factual premise."))
    for n in names:
        if n.startswith("RuleTaker") and RESULTS[n]["errors"]:
            e = RESULTS[n]["errors"][0]; st += [P(f"Example error ({esc(n)}): statement <i>{esc(str(e['query']))}</i>, gold {e['gold']}, system {e['pred']}"), Preformatted(str(e["context"])[:700], MONO)]; break
    st += [P("4. Reasoning traces", H1)]
    st += trace_block("Assignment example", "Bob is young. Bob is kind. If someone is young, then they are smart. If someone is smart and kind, then they are successful.", "Bob is successful.")
    st += trace_block("RuleTaker (course format)", "Facts:\nLiam volunteers.\nRules:\nWhen liam volunteers, we can conclude liam hikes mountains.\nGiven that liam hikes mountains, it follows that henry volunteers.", "Henry volunteers.")
    st += trace_block("FOLIO (resolution)", "∀x (Man(x) → Mortal(x))\nMan(socrates)", "Mortal(socrates)")
    st += trace_block("CounterBench, answer No (do(Ziklo=0))", "", CB_EX1)
    st += trace_block("CounterBench, answer Yes (do(Nuv=0, Splee=0); NAF on Druk)", "", CB_DEMO)
    st += [P("5. Limitations", H1), P("English FOLIO premises without logic annotations are outside the rule parser&rsquo;s coverage (the task supplies the formulas). Equality is treated as an ordinary predicate (no substitution axioms). "
              "Resolution is refutation-complete but resource-bounded (time, clause count, term depth), so hard instances end as <i>not entailed</i>. Routing uses surface features: English text that contains FOL symbols and parentheses would be sent to the FOL prover. "
              "The CounterBench parser is template-based; Prolog-style rules and LaTeX notation are not supported.")]
    SimpleDocTemplate(path, pagesize=A4, leftMargin=42, rightMargin=42, topMargin=40, bottomMargin=40).build(st)
    print("wrote", path)
try: make_report()
except ImportError: print("pip install reportlab to generate report.pdf")

wrote report.pdf
